# Stable Log-Sum-Exp, Softmax, and Cross-Entropy

## Objective
Implement numerically stable classification calculations and examine overflow, underflow, and common-shift behavior.

## Stable Formulas
Let \(m = \max_i z_i\).

$$
\operatorname{LSE}(z)
= m + \ln\left(\sum_i e^{z_i-m}\right)
$$

$$
p_i = \frac{e^{z_i-m}}{\sum_j e^{z_j-m}}
$$

For correct-class index \(y\):

$$
L = \operatorname{LSE}(z)-z_y.
$$

## Validation

| Logits | Expected LSE | Cross-entropy, target index 1 |
|---|---:|---:|
| `[1000, 1000]` | ≈1000.693147 | ≈0.693147 |
| `[0, -1000]` | ≈0 | ≈1000 |
| `[-1000, -1000]` | ≈−999.306853 | ≈0.693147 |

Adding the same constant to every logit adds that constant to LSE while leaving softmax probabilities and cross-entropy unchanged.

## Numerical Stability
Maximum subtraction prevents exponential overflow for finite logits. Computing cross-entropy directly from logits avoids taking the logarithm of a tiny probability that has rounded to zero.

## Scope
The finite-input practical passed after correcting the cross-entropy subtraction direction with guidance. It does not reproduce the paper’s full numerical error analysis.

## Reference
*Accurate Computation of the Log-Sum-Exp and Softmax Functions*.

In [4]:
fn logsumexp(x: &[f64]) -> f64 {
    
    if x.is_empty(){
        return f64::NEG_INFINITY;
    }
    let c = x.iter().copied().fold(f64::NEG_INFINITY, f64::max);

    if c.is_infinite() && c.is_sign_negative(){
        return f64::NEG_INFINITY;
    }

    let sum_exp: f64 = x.iter().map(|&xi| (xi-c).exp()).sum();
    c + sum_exp.ln()

}

In [11]:
// --- Test: Naive vs Stable ---
let logits_1 = vec![1000.0,1000.0];
let logits_2 = vec![0.0,-1000.0];
let logits_3 = vec![-1000.0,-1000.0];


// Stable LogSumExp
let stable1 = logsumexp(&logits_1);
let stable2 = logsumexp(&logits_2);
let stable3 = logsumexp(&logits_3);

println!("Stable: {},{},{}", stable1,stable2,stable3); // Output: 1002.4076059644443

Stable: 1000.6931471805599,0,-999.3068528194401


In [ ]:
fn cross_entropy(x: &[f64], target_class: usize) -> f64 {
    
    if x.is_empty() || target_class >= x.len(){
        return f64::NAN;
    }
    let correct_logit = x[target_class];
    let lse = logsumexp(x);

    lse - correct_logit
}

In [12]:
// Stable LogSumExp
let stable1 = softmax(&logits_1, 1);
let stable2 = softmax(&logits_2, 1);
let stable3 = softmax(&logits_3, 1);

println!("Stable: {},{},{}", stable1,stable2,stable3); // Output: 1002.4076059644443

Stable: 0.6931471805598903,1000,0.6931471805598903


In [18]:
fn softmax(logits: &[f64]) -> Vec<f64>{

    if logits.is_empty() {
        return Vec::new();
    }

    let max_logit = logits.iter().copied().fold(f64::NEG_INFINITY, f64::max);

    let exp: Vec<f64> = logits.iter().map(|&xi| (xi-max_logit).exp()).collect();
    let exp_sum: f64 = exp.iter().sum();

    exp.iter().map(|&x| x/exp_sum).collect()
}

In [23]:
fn main() {
    let cases: Vec<Vec<f64>> = vec![
        vec![1000.0, 1000.0],
        vec![1000.0, 1001.0, 1002.0],
        vec![-1000.0, -1000.0],
    ];

    for logits in &cases {
        let probabilities = softmax(logits);
        let total: f64 = probabilities.iter().sum();

        println!("Input: {:?}", logits);
        println!("Probabilities: {:?}", probabilities);
        println!("Sum: {:.6}\n", total);
    }
}

In [24]:
main()

Input: [1000.0, 1000.0]
Probabilities: [0.5, 0.5]
Sum: 1.000000

Input: [1000.0, 1001.0, 1002.0]
Probabilities: [0.09003057317038046, 0.24472847105479764, 0.6652409557748218]
Sum: 1.000000

Input: [-1000.0, -1000.0]
Probabilities: [0.5, 0.5]
Sum: 1.000000



()